# Tabular Classification Dataset Analysis

This notebook performs an exploratory analysis on a tabular classification dataset. We will load the data, examine its structure, and visualize the class distribution.

In [ ]:
# Install missing libraries in the current kernel if needed
try:
    import pandas, seaborn, matplotlib, sklearn
except ImportError:
    %pip install pandas seaborn matplotlib scikit-learn

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Set seed for reproducibility
RANDOM_STATE = 42

In [ ]:
# Record library versions for reproducibility
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
try:
    print(f"Random State: {RANDOM_STATE}")
except NameError:
    print("Random State: Not defined yet")
print("="*60)

## 1. Load Dataset

In [ ]:
# Loading the local CSV dataset
df = pd.read_csv('my_data .csv')

# Split features (X) and target (y)
# Assuming 'placed' is the target column based on the dataset structure
X = df.drop(columns=['placed'])
y = df['placed']

print("Dataset loaded successfully.")

## 2. Dataset Overview

In [ ]:
# 1) Shape of X and y
print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

In [ ]:
# 2) List of feature names and dtypes
print("\nFeature Names and Data Types:")
print(X.dtypes)

In [ ]:
# Identifying numerical and categorical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical Features: {num_features}")
print(f"Categorical Features: {cat_features}")

## 3. Class Distribution

In [ ]:
# 3) Class distribution (counts and percentages)
counts = y.value_counts()
percentages = y.value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages
})
print("\nClass Distribution:")
print(dist_df)

## 4. Visualization

In [ ]:
# 4) Visualization: Class Count Plot
plt.figure(figsize=(8, 5))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Target Class (Placed)')
plt.xlabel('Placed (0: No, 1: Yes)')
plt.ylabel('Count')
plt.show()

## 5. Train/Test Split

In [ ]:
# Stratified train/test split (25% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"\nStratified Split completed:")
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Verify stratification
print("\nTraining Class Distribution (%):")
print(y_train.value_counts(normalize=True) * 100)
print("\nTesting Class Distribution (%):")
print(y_test.value_counts(normalize=True) * 100)

## 6. Preprocessing Pipeline

In [ ]:
# Define transformers for numerical and categorical features
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers via ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

# Fit on train and transform both train and test
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Transformed Training size: {X_train_transformed.shape}")
print(f"Transformed Testing size: {X_test_transformed.shape}")

### Preprocessing Justification
Median imputation is appropriate here as it is robust to outliers, ensuring that missing values don't skew the distribution. Standardization is essential for distance-based (e.g., KNN) or gradient-based models (e.g., Logistic Regression) to ensure all features are on a comparable scale, preventing features with larger magnitudes from dominating the learning process. MinMaxScaler would be preferable if the model requires features to be within a specific range (like 0 to 1 for Neural Networks with sigmoid activations) or if the distribution is not Gaussian and we want to preserve the relative scaling without assuming zero mean and unit variance.